In [ ]:
import pandas as pd
df = pd.read_csv("combined_2016_2020_precincts.csv")

In [ ]:
data = df[['PRECINCTID', 'Precinct_L', 'age_18_19', 'age_20_24',
       'age_25_29', 'age_35_44', 'age_45_54', 'age_55_64', 'age_65_74',
       'age_75_84', 'age_85over', 'party_dem', 'party_rep', 'party_npp',
       'eth1_eur', 'eth1_hisp', 'eth1_aa', 'eth1_esa', 'eth1_oth']]

size = 1000
data = data[:size]


In [ ]:
dat2 = df[:size]

In [ ]:
import geopandas as gpd
import os
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt

from scipy.spatial import distance
from sklearn.metrics import pairwise_distances
from scipy.stats import entropy
import numpy as np



AGE_CATEGORIES = ['age_18_19', 'age_20_24' ,'age_25_29', 'age_35_44', 'age_45_54', 'age_55_64', 'age_65_74', 'age_75_84', 'age_85over']
PARTY_ID_CATEGORIES = ['party_dem', 'party_rep', 'party_npp']
DEMOGRAPHIC_CATEGORIES = ['eth1_eur', 'eth1_hisp', 'eth1_aa', 'eth1_esa', 'eth1_oth']


total_filters = ['PRECINCTID', 'Precinct_L'] + AGE_CATEGORIES + PARTY_ID_CATEGORIES + DEMOGRAPHIC_CATEGORIES

all_categorical_features = AGE_CATEGORIES + PARTY_ID_CATEGORIES + DEMOGRAPHIC_CATEGORIES

# Note: in MI eth1_oth is likely arab voters
DEMOGRAPHIC_CATEGORIES = ['eth1_eur', 'eth1_hisp', 'eth1_aa', 'eth1_esa', 'eth1_oth']

# Function to calculate Jensen-Shannon Divergence
def jensen_shannon_divergence(p, q):
    m = 0.5 * (p + q)
    return 0.5 * (entropy(p, m) + entropy(q, m))

# Compute pairwise Jensen-Shannon Divergence for each category
def compute_category_distances(data, categories):
    # Normalize data if not already percentages
    data_normalized = data[categories].div(data[categories].sum(axis=1), axis=0)
    dist_matrix = pairwise_distances(data_normalized, metric=lambda x, y: jensen_shannon_divergence(x, y))
    return dist_matrix

# Compute distances for each category
age_distances = compute_category_distances(data, AGE_CATEGORIES)
party_distances = compute_category_distances(data, PARTY_ID_CATEGORIES)
demographic_distances = compute_category_distances(data, DEMOGRAPHIC_CATEGORIES)

# Aggregate these distances (simple average)
total_distances = (age_distances + party_distances + demographic_distances) / 3

# Find nearest neighbors manually
def find_nearest_neighbors(dist_matrix, index, n_neighbors=10):
    distances = dist_matrix[index]
    valid_indices = np.where(distances != 0)[0]  # Get indices where distance is not 0
    sorted_indices = np.argsort(distances[valid_indices])  # Sort based on distances, ignoring zeroes
    nearest_indices = valid_indices[sorted_indices][:n_neighbors]  # Get the top n_neighbors
    return nearest_indices, distances[nearest_indices]

In [ ]:
import numpy as np

class Precinct:
    def __init__(self, id, estimated_dem, estimated_rep, estimated_turnout, similar_precincts):
        """
        Initializes a precinct.
        :param id: Precinct ID
        :param estimated_dem: Estimated Democratic vote share
        :param estimated_rep: Estimated Republican vote share
        :param estimated_turnout: Estimated turnout
        :param similar_precincts: List of tuples with similar precincts (id, distance)
        """
        self.id = id
        self.estimated_dem = estimated_dem
        self.estimated_rep = estimated_rep
        self.estimated_turnout = estimated_turnout
        self.similar_precincts = similar_precincts
        self.reported = False
        self.actual_dem = None
        self.actual_rep = None
        self.actual_turnout = None
        self.adjusted_dem = estimated_dem
        self.adjusted_rep = estimated_rep
        self.adjusted_turnout = estimated_turnout

def sigmoid(x):
    """ Sigmoid function for scaling adjustments based on the reporting rate """
    return 1 / (1 + np.exp(-x))

def update_precinct_results(precinct, actual_dem, actual_rep, actual_turnout):
    """ Update precinct with actual results """
    precinct.reported = True
    precinct.actual_dem = actual_dem
    precinct.actual_rep = actual_rep
    precinct.actual_turnout = actual_turnout

def calculate_adjustments(precincts, hyper_parameter = 1):
    """ Adjust the baseline predictions based on reported results from similar precincts """
    for precinct in precincts:
        if not precinct.reported:
            total_weight = 0
            dem_deviation_sum = 0
            rep_deviation_sum = 0
            turnout_deviation_sum = 0

            for similar_id, distance in precinct.similar_precincts:
                similar_precinct = next((p for p in precincts if p.id == similar_id), None)
                if similar_precinct and similar_precinct.reported and distance > 0:  # Ensure valid distance
                    weight = 1 / distance  # Weight inversely proportional to the distance
                    dem_deviation = similar_precinct.actual_dem - similar_precinct.estimated_dem
                    rep_deviation = similar_precinct.actual_rep - similar_precinct.estimated_rep
                    turnout_deviation = similar_precinct.actual_turnout - similar_precinct.estimated_turnout
                    dem_deviation_sum += dem_deviation * weight
                    rep_deviation_sum += rep_deviation * weight
                    turnout_deviation_sum += turnout_deviation * weight
                    total_weight += weight

            if total_weight > 0:
                reporting_factor = hyper_parameter # Apply sigmoid to scale the impact
                precinct.adjusted_dem += (dem_deviation_sum / total_weight) * reporting_factor
                precinct.adjusted_rep += (rep_deviation_sum / total_weight) * reporting_factor
                precinct.adjusted_turnout += (turnout_deviation_sum / total_weight) * reporting_factor

In [ ]:
import random

def randomly_report_precincts(precincts, data, report_percentage=0.3):
    """
    Randomly chooses 30% of the precincts and reports their actual data from the 2020 columns.
    
    :param precincts: List of Precinct objects
    :param data: DataFrame containing actual results with 2020 column names
    :param report_percentage: Percentage of precincts to report (default is 30%)
    """
    num_precincts = len(precincts)
    num_to_report = int(num_precincts * report_percentage)

    # Randomly select 30% of the precincts
    selected_indices = random.sample(range(num_precincts), num_to_report)
    
    for i in selected_indices:
        precinct = precincts[i]
        
        # Extract actual data for this precinct from the 2020 columns
        actual_dem = dat2.loc[precinct.id, 'DEM VOTE - 2020']  # Assuming '2020_actual_dem' column
        actual_rep = dat2.loc[precinct.id, 'REP VOTE - 2020']  # Assuming '2020_actual_rep' column
        actual_turnout = dat2.loc[precinct.id, 'TURNOUT - 2020']  # Assuming '2020_actual_turnout' column
        
        # Report the actual results
        update_precinct_results(precinct, actual_dem, actual_rep, actual_turnout)
        




In [ ]:
def calculate_predicted_results(precincts):
    """
    Calculate the predicted election results by aggregating Democratic and Republican votes 
    across all precincts, using actual results if reported, otherwise using estimated results.
    
    :param precincts: List of Precinct objects
    :return: Predicted total Democratic votes, Republican votes
    """
    total_dem_votes = 0
    total_rep_votes = 0

    for precinct in precincts:
        if precinct.reported:
            # Use actual reported results if precinct has reported
            dem_votes = precinct.actual_dem * precinct.actual_turnout
            rep_votes = precinct.actual_rep * precinct.actual_turnout
        else:
            # Use estimated results if precinct has not reported
            dem_votes = precinct.adjusted_dem * precinct.adjusted_turnout
            rep_votes = precinct.adjusted_rep * precinct.adjusted_turnout

        # Aggregate the votes
        total_dem_votes += dem_votes
        total_rep_votes += rep_votes

    return total_dem_votes, total_rep_votes

def calculate_actual_results(dat2):
    """
    Aggregate the actual 2020 election results from the 'dat2' DataFrame, where 
    '2020_actual_dem_votes' and '2020_actual_rep_votes' are proportions.
    
    :param dat2: DataFrame containing true 2020 election results with proportions 
                 for Democratic and Republican votes, and total turnout
    :return: Total Democratic votes, Republican votes
    """
    # Calculate actual votes by multiplying proportions by the total turnout for each precinct
    total_actual_dem_votes = (dat2['DEM VOTE - 2020'] * dat2['TURNOUT - 2020']).sum()
    total_actual_rep_votes = (dat2['REP VOTE - 2020'] * dat2['TURNOUT - 2020']).sum()

    return total_actual_dem_votes, total_actual_rep_votes

def calculate_mape(predicted_dem, predicted_rep, actual_dem, actual_rep):
    """
    Calculate the mean absolute percentage error (MAPE) between predicted and actual results
    for Democratic and Republican votes.
    
    :param predicted_dem: Total predicted Democratic votes
    :param predicted_rep: Total predicted Republican votes
    :param actual_dem: Total actual Democratic votes
    :param actual_rep: Total actual Republican votes
    :return: MAPE (mean absolute percentage error)
    """
    dem_mape = abs((actual_dem - predicted_dem) / actual_dem) * 100
    rep_mape = abs((actual_rep - predicted_rep) / actual_rep) * 100
    overall_mape = (dem_mape + rep_mape) / 2
    return overall_mape

def compare_predicted_to_actual(precincts, dat2):
    """
    Compare the predicted election results to the actual 2020 election results and 
    calculate the mean absolute percentage error (MAPE) for Democratic and Republican votes.
    
    :param precincts: List of Precinct objects with predicted results
    :param dat2: DataFrame containing actual 2020 election results
    """
    # Calculate predicted results using the precinct objects
    predicted_dem, predicted_rep = calculate_predicted_results(precincts)
    
    # Calculate actual results using the dat2 DataFrame
    actual_dem, actual_rep = calculate_actual_results(dat2)
    
    # Calculate predicted and actual percentages
    total_predicted_votes = predicted_dem + predicted_rep
    total_actual_votes = actual_dem + actual_rep

    predicted_dem_percentage = (predicted_dem / total_predicted_votes) * 100
    predicted_rep_percentage = (predicted_rep / total_predicted_votes) * 100
    
    actual_dem_percentage = (actual_dem / total_actual_votes) * 100
    actual_rep_percentage = (actual_rep / total_actual_votes) * 100
    
    # Calculate MAPE between predicted and actual results
    mape = calculate_mape(predicted_dem, predicted_rep, actual_dem, actual_rep)
    
    # Output the comparison of predicted vs. actual results
    print("Comparison of Predicted vs. Actual Results:")
    print(f"Predicted Democratic Votes: {predicted_dem:.0f} ({predicted_dem_percentage:.2f}%)")
    print(f"Predicted Republican Votes: {predicted_rep:.0f} ({predicted_rep_percentage:.2f}%)")
    
    print("\nActual 2020 Results:")
    print(f"Actual Democratic Votes: {actual_dem:.0f} ({actual_dem_percentage:.2f}%)")
    print(f"Actual Republican Votes: {actual_rep:.0f} ({actual_rep_percentage:.2f}%)")
    
    # Output the MAPE (mean absolute percentage error)
    print(f"\nMean Absolute Percentage Error (MAPE): {mape:.2f}%")
    return mape




In [ ]:
nearest_neighbors_info = []

for i in range(size):
    indices, dists = find_nearest_neighbors(total_distances, i, size)
    neighbors_info = {'precinct': i, 'nearest_indices': indices, 'distances': dists}
    nearest_neighbors_info.append(neighbors_info)


precinct_objects = []

for info in nearest_neighbors_info:
    precinct_id = info['precinct']
    
    # Retrieve estimated values from the dataset for each precinct
    estimated_dem = dat2.loc[precinct_id, 'DEM VOTE - 2016']
    estimated_rep = dat2.loc[precinct_id, 'REP VOTE - 2016']
    estimated_turnout = dat2.loc[precinct_id, 'TURNOUT - 2016']
    
    # Get nearest precincts and distances
    similar_precincts = list(zip(info['nearest_indices'], info['distances']))
    
    # Create a Precinct object
    precinct = Precinct(
        id=precinct_id,
        estimated_dem=estimated_dem,
        estimated_rep=estimated_rep,
        estimated_turnout=estimated_turnout,
        similar_precincts=similar_precincts
    )
    
    # Add to the precinct_objects list
    precinct_objects.append(precinct)

# Example Usage:
# Assuming you have created a list of Precinct objects (precinct_objects) and the data DataFrame with 2020 results.
randomly_report_precincts(precinct_objects, data, report_percentage=0.3)

calculate_adjustments(precinct_objects, hyper_parameter = 1)

# Example Usage:
# Assuming you have created a list of Precinct objects (precinct_objects) and the dat2 DataFrame with actual 2020 results
mape = compare_predicted_to_actual(precinct_objects, dat2)

In [ ]:
print(mape)

In [ ]:
def instantiate_precinct_objects(nearest_neighbors_info, dat2):
    """ Re-instantiate the precinct objects from nearest neighbors info and data. """
    precinct_objects = []
    for info in nearest_neighbors_info:
        precinct_id = info['precinct']
        
        # Retrieve estimated values from the dataset for each precinct
        estimated_dem = dat2.loc[precinct_id, 'DEM VOTE - 2016']
        estimated_rep = dat2.loc[precinct_id, 'REP VOTE - 2016']
        estimated_turnout = dat2.loc[precinct_id, 'TURNOUT - 2016']
        
        # Get nearest precincts and distances
        similar_precincts = list(zip(info['nearest_indices'], info['distances']))
        
        # Create a Precinct object
        precinct = Precinct(
            id=precinct_id,
            estimated_dem=estimated_dem,
            estimated_rep=estimated_rep,
            estimated_turnout=estimated_turnout,
            similar_precincts=similar_precincts
        )
        
        # Add to the precinct_objects list
        precinct_objects.append(precinct)
    
    return precinct_objects

def test_hyperparameters(nearest_neighbors_info, dat2, data, report_percentage=0.1, trials=20):
    """
    Test different hyperparameters between 0 and 1 and return the corresponding average MAPE for each hyperparameter.
    
    :param nearest_neighbors_info: Information to build the precinct objects
    :param dat2: DataFrame containing actual and estimated 2016/2020 results
    :param data: DataFrame for actual reporting
    :param report_percentage: Percentage of precincts to randomly report
    :param trials: Number of times to test each hyperparameter
    :return: Dictionary of hyperparameters and their corresponding average MAPE
    """
    hyperparameter_results = {}

    # Range of hyperparameters between 0 and 1
    hyperparameter_values = np.linspace(0.85, 0.95, 11)

    for hyperparameter in hyperparameter_values:
        mape_values = []
        
        # Test each hyperparameter value over multiple trials
        for _ in range(trials):
            # Re-instantiate precinct objects for each trial
            precinct_objects = instantiate_precinct_objects(nearest_neighbors_info, dat2)
            
            # Randomly report precincts
            randomly_report_precincts(precinct_objects, data, report_percentage)
            
            # Calculate adjustments using the current hyperparameter
            calculate_adjustments(precinct_objects, hyper_parameter=hyperparameter)
            
            # Calculate MAPE
            mape = compare_predicted_to_actual(precinct_objects, dat2)
            mape_values.append(mape)

        # Calculate average MAPE for this hyperparameter
        average_mape = np.mean(mape_values)
        
        # Store the hyperparameter and its average MAPE
        hyperparameter_results[hyperparameter] = average_mape

    return hyperparameter_results

# Example Usage:
# Assuming you have created the nearest_neighbors_info and the dat2 DataFrame
hyperparameter_results = test_hyperparameters(nearest_neighbors_info, dat2, data)

# Print the hyperparameters and their corresponding average MAPE
for hyperparameter, average_mape in hyperparameter_results.items():
    print(f"Hyperparameter: {hyperparameter:.2f}, Average MAPE: {average_mape:.4f}")

In [ ]:
def calculate_projected_2016_results(dat2):
    """
    Calculate projected 2020 election results if 2016 Democratic and Republican vote proportions
    held exactly the same in 2020.
    
    :param dat2: DataFrame containing 2016 vote proportions and 2020 total turnout
    :return: Projected total Democratic votes, Republican votes based on 2016 proportions
    """
    # Use 2016 proportions multiplied by 2020 total turnout to project 2020 votes
    projected_dem_votes = (dat2['DEM VOTE - 2016'] * dat2['TURNOUT - 2020']).sum()
    projected_rep_votes = (dat2['REP VOTE - 2016'] * dat2['TURNOUT - 2020']).sum()
    
    return projected_dem_votes, projected_rep_votes

def compare_projected_2016_to_actual(dat2):
    """
    Compare projected 2020 election results (if 2016 proportions held) to the actual 2020 results.
    
    :param dat2: DataFrame containing 2016 vote proportions and actual 2020 results
    """
    # Calculate projected results assuming 2016 proportions hold
    projected_dem, projected_rep = calculate_projected_2016_results(dat2)
    
    # Calculate actual results from the dat2 DataFrame
    actual_dem, actual_rep = calculate_actual_results(dat2)
    
    # Calculate projected and actual percentages
    total_projected_votes = projected_dem + projected_rep
    total_actual_votes = actual_dem + actual_rep

    projected_dem_percentage = (projected_dem / total_projected_votes) * 100
    projected_rep_percentage = (projected_rep / total_projected_votes) * 100
    
    actual_dem_percentage = (actual_dem / total_actual_votes) * 100
    actual_rep_percentage = (actual_rep / total_actual_votes) * 100
    
    # Output the comparison of projected 2016 results vs. actual 2020 results
    print("Comparison of Projected 2016 Results (Applied to 2020 Turnout) vs. Actual 2020 Results:")
    print(f"Projected Democratic Votes (2016 Proportions): {projected_dem:.0f} ({projected_dem_percentage:.2f}%)")
    print(f"Projected Republican Votes (2016 Proportions): {projected_rep:.0f} ({projected_rep_percentage:.2f}%)")
    
    print("\nActual 2020 Results:")
    print(f"Actual Democratic Votes: {actual_dem:.0f} ({actual_dem_percentage:.2f}%)")
    print(f"Actual Republican Votes: {actual_rep:.0f} ({actual_rep_percentage:.2f}%)")
    
compare_projected_2016_to_actual(dat2)


In [ ]:
dat2.columns